In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
masterPipelineRunID = '1234'

In [0]:
configGoldDF = spark.read \
  .format("jdbc") \
  .option("url", f"jdbc:sqlserver://insuranceserver2002.database.windows.net:1433;database=insuarnce") \
  .option("dbtable", "insurance.gold_Config") \
  .option("user",'user') \
  .option("password", 'Admin1234') \
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
  .load()

In [0]:
def getColumnsForMerge(ObjectName, sourceAlias="s", targetAlias="t", skipCols=None):
    if skipCols is None:
        skipCols = [ "MODIFIED_ON", "MODIFIED_BY"]

    row = (
        configGoldDF
        .filter(col("OBJECT_NAME") == ObjectName).orderBy("ID","GOLD_COLUMN_NAMES")
        .select("GOLD_COLUMN_NAMES")
        .first()
    )

    if not row:
        raise ValueError(f"No mappings found for {ObjectName}")

    cols = [c.strip() for c in row["GOLD_COLUMN_NAMES"].split(",")]
    
    src_columns = []
    tgt_columns = []
    merge_pairs = []

    for c in cols:
        src_columns.append(f"{sourceAlias}.{c}")
        tgt_columns.append(c)

        if c not in skipCols and c != "ACCOUNT_KEY":
            merge_pairs.append(f"{targetAlias}.{c} = {sourceAlias}.{c}")
    cols_to_remove = ["ADDED_BY","ADDED_ON","MODIFIED_BY", "MODIFIED_ON"]
    for col_name in cols_to_remove:
        aliased_col = f"{sourceAlias}.{col_name}"
        if aliased_col in src_columns:
            src_columns.remove(aliased_col)
    sourceColumn = ", ".join(src_columns)
    targetColumn = ", ".join(tgt_columns)
    mergeColumnStatement = ", ".join(merge_pairs)
    insertSourceColumn = sourceColumn + ", s.ADDED_BY, s.ADDED_ON, s.MODIFIED_BY, s.MODIFIED_ON"
    
    return sourceColumn, insertSourceColumn,targetColumn, mergeColumnStatement


In [0]:
def load_data_into_gold(objectName ,targetTableName, finalDF, operationType, keyColumnName,targetCatalogName,targetSchemaName,isFullLoad,masterPipelineRunID):
    
    finalDF.createOrReplaceTempView("tempView")
    srcColumns,insertSourceColumn,targetColumns,mergeColumnStatement = getColumnsForMerge(ObjectName, sourceAlias="s", targetAlias="t", skipCols=None)
    if operationType.lower() == "upsert":
            if isFullLoad == 1:
                print(f"FULL LOAD STARTED FOR {objectName}")
                delete_query = spark.sql(f"DELETE FROM {targetCatalogName}.{targetSchemaName}.{targetTableName}")
                insert_query = spark.sql(f"""
                                    INSERT INTO {targetCatalogName}.{targetSchemaName}.{targetTableName} ({targetColumns})
                                    SELECT {insertSourceColumn}
                                    FROM tempView s
                                    """)
                df_history = spark.sql(f"describe history {targetCatalogName}.{targetSchemaName}.{targetTableName} limit 1").first()
                rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                print("Number of Rows Inserted :", rowsInserted)
                print(f"Data Loading for {objectName} completed")
            else:
                print(f"Merge Operation Started on {targetTableName}")
                mergeQuery = f"""
                MERGE INTO {targetCatalogName}.{targetSchemaName}.{targetTableName} t
                USING tempView s ON s.{keyColumnName}=t.{keyColumnName}
                WHEN MATCHED THEN UPDATE
                SET {mergeColumnStatement},Modified_On = CURRENT_TIMESTAMP, Modified_By = '{masterPipelineRunID}'
                WHEN NOT MATCHED THEN INSERT ({targetColumns}) VALUES ({srcColumns},CURRENT_TIMESTAMP,'{masterPipelineRunID}', CURRENT_TIMESTAMP, '{masterPipelineRunID}')
                """
                print("Executing merge query")
                spark.sql(mergeQuery)
                print(f"Data Loading for {objectName} completed")
                
    

In [0]:
def dim_penetration_rates():
    penetrationRatesDF = spark.read.table('insurancetej.silver.penetrationrates')\
                .select(col('ROWSEQUENCE_NFIPRESIDENTIALPENETRATIONRATES').alias('PENETRATION_RATES_CODE')\
                ,col('METADATA_URL').alias('PENETRATION_RATES_URL')
                ,col('NFIPRESIDENTIALPENETRATIONRATES_COUNTY').alias('PENETRATION_RATES_COUNTY')
                ,col('NFIPRESIDENTIALPENETRATIONRATES_TOTALRESSTRUCTURES').alias('PENETRATION_RATES_TOTAL_RESTRUCTURES')
                ,col('NFIPRESIDENTIALPENETRATIONRATES_STATE').alias('PENETRATION_RATES_STATE'))
    finalDF = penetrationRatesDF.withColumn('PENETRATION_RATES_KEY',row_number().over(Window.orderBy(col('PENETRATION_RATES_CODE'))))
    finalDF = finalDF.withColumn('ADDED_ON',current_timestamp())\
                     .withColumn('ADDED_BY',lit(masterPipelineRunID))\
                     .withColumn('MODIFIED_ON',current_timestamp())\
                     .withColumn('MODIFIED_BY',lit(masterPipelineRunID))
    targetTableDF = spark.read.table("insurancetej.gold.dim_penetration_rates")
    targetSchema = targetTableDF.schema
    finalDF = finalDF.select(*[ col(field.name).cast(field.dataType).alias(field.name) for field in targetSchema])
    return finalDF

In [0]:
def fact_claims():
    claimsDF = spark.read.table('insurancetej.silver.claims')\
                .select(col('ROWSEQUENCE_FIMANFIPCLAIMS').alias('CLAIMS_CODE')\
                ,col('METADATA_ENTITYNAME').alias('CLAIMS_ENTITY_NAME'),col('FIMANFIPCLAIMS_ID').alias('CLAIM_ID'),'FIMANFIPCLAIMS_CONTENTSDAMAGEAMOUNT','FIMANFIPCLAIMS_CONTENTSPROPERTYVALUE')
    finalDF = claimsDF.withColumn('CLAIMS_KEY',row_number().over(Window.orderBy(col('CLAIMS_CODE'))))
    finalDF = finalDF.withColumn("CLAIM_SETTLEMENT_CASE",when(col("FIMANFIPCLAIMS_CONTENTSDAMAGEAMOUNT") > 50000, "NEGATIVE").otherwise("POSITIVE"))\
        .withColumn("CLAIM_APPROVAL_STATUS",when(col("FIMANFIPCLAIMS_CONTENTSPROPERTYVALUE") > 80000, "NOT APPROVED")
    .otherwise("APPROVED"))
    finalDF = finalDF.withColumn('ADDED_ON',current_timestamp())\
                     .withColumn('ADDED_BY',lit(masterPipelineRunID))\
                     .withColumn('MODIFIED_ON',current_timestamp())\
                     .withColumn('MODIFIED_BY',lit(masterPipelineRunID))
    targetTableDF = spark.read.table("insurancetej.gold.fact_claims")
    targetSchema = targetTableDF.schema
    finalDF = finalDF.select(*[ col(field.name).cast(field.dataType).alias(field.name) for field in targetSchema])
    return finalDF

In [0]:
try:
    for row in configGoldDF.collect():
            ObjectName = row['OBJECT_NAME']
            sourceName = 'api'
            targetTableName = row['OBJECT_NAME']
            operationType = row['OPERATION_TYPE']
            keyColumnName = row['GOLD_KEY_COLUMN_NAME']
            targetCatalogName = row['GOLD_CATALOG_NAME']
            targetSchemaName = row['GOLD_SCHEMA_NAME']
            isFullLoad = row['IS_HISTORICAL']
            if ObjectName.lower() == 'dim_penetration_rates':
                finalDF = dim_penetration_rates()
                load_data_into_gold(ObjectName ,targetTableName, finalDF, operationType, keyColumnName,targetCatalogName,targetSchemaName,isFullLoad,masterPipelineRunID)
            elif ObjectName.lower() == 'fact_claims':
                finalDF = fact_claims()
                load_data_into_gold(ObjectName ,targetTableName, finalDF, operationType, keyColumnName,targetCatalogName,targetSchemaName,isFullLoad,masterPipelineRunID)
            else:
                print(f"Invalid Object Name {ObjectName}")
except Exception as e:
    print(f"Error occurred: {e}")
    raise